# Basic


In [ ]:
# install packages
# this notebook uses a zero-shot nli model as the metatox-inspired branch

!pip uninstall -y -q sentence-transformers torchvision torchaudio torchtext timm

!pip install -q --no-cache-dir --upgrade --no-deps \
    "transformers==4.51.3" \
    "tokenizers==0.21.1" \
    "huggingface-hub==0.34.4" \
    "safetensors==0.5.3"

!pip install -q tqdm joblib sentencepiece protobuf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 108.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 244.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 129.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 264.7 MB/s eta 0:00:00


In [ ]:
# basic imports
import os
from pathlib import Path
import random

# avoid unnecessary backend imports
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

from google.colab import drive
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [ ]:
# quick environment check
import importlib.util
import transformers

print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('cuda available:', torch.cuda.is_available())

print('timm:', importlib.util.find_spec('timm'))
print('torchvision:', importlib.util.find_spec('torchvision'))

torch: 2.11.0+cu128
transformers: 4.51.3
cuda available: True
timm: None
torchvision: None


In [ ]:
# experiment settings
seed = 42

model_name = 'MoritzLaurer/deberta-v3-base-zeroshot-v2.0'

batch_size = 16
max_len = 512

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('device:', device)
print('model:', model_name)

if device == 'cpu':
    print('warning: this will be slow on cpu. gpu runtime is better.')

device: cuda
model: MoritzLaurer/deberta-v3-base-zeroshot-v2.0


In [ ]:
# mount drive and set paths
drive.mount('/content/drive')

project_dir = Path('/content/drive/MyDrive/Colab Notebooks/Project')

td_dir = project_dir / 'toxicdetector_style_qwen3_outputs'
output_dir = project_dir / 'metatox_inspired_zeroshot_outputs'

output_dir.mkdir(parents=True, exist_ok=True)

print('toxicdetector folder:', td_dir)
print('output folder:', output_dir)

Mounted at /content/drive
toxicdetector folder: /content/drive/MyDrive/Colab Notebooks/Project/toxicdetector_style_qwen3_outputs
output folder: /content/drive/MyDrive/Colab Notebooks/Project/metatox_inspired_zeroshot_outputs


# Dataset

In [ ]:
# load common val and test data from notebook 01
val_df = pd.read_csv(td_dir / 'common_val_hx_jigsaw.csv')
test_df = pd.read_csv(td_dir / 'common_test_hx_jigsaw.csv')

val_df = val_df.dropna(subset=['text', 'label']).reset_index(drop=True)
test_df = test_df.dropna(subset=['text', 'label']).reset_index(drop=True)

val_df['text'] = val_df['text'].astype(str)
test_df['text'] = test_df['text'].astype(str)

val_df['label'] = val_df['label'].astype(int)
test_df['label'] = test_df['label'].astype(int)

print('val:', val_df.shape)
print('test:', test_df.shape)

print('\nval source count')
print(val_df['source'].value_counts())

print('\ntest source count')
print(test_df['source'].value_counts())

val_df.head()

val: (25858, 4)
test: (25860, 4)

val source count
source
jigsaw        23936
hatexplain     1922
Name: count, dtype: int64

test source count
source
jigsaw        23936
hatexplain     1924
Name: count, dtype: int64


,sample_id,text,label,source
0,jg_36797,"""\n\nShall you immediately stop this sock-pupp...",0,jigsaw
1,hx_validation_341,this is what <number> years of kike brainwashi...,1,hatexplain
2,hx_validation_1510,non asians btw u cant make jokes like “ why do...,0,hatexplain
3,jg_120449,I don't really think renaming should be a cent...,0,jigsaw
4,jg_112467,Apologies \n\nfor the stupid pointless bickeri...,0,jigsaw


In [ ]:
# check toxicdetector outputs from notebook 01
td_val = pd.read_csv(td_dir / 'toxicdetector_style_val_outputs.csv')
td_test = pd.read_csv(td_dir / 'toxicdetector_style_outputs.csv')

print('td val:', td_val.shape)
print('td test:', td_test.shape)

val_same_order = (
    val_df['sample_id'].astype(str).values
    == td_val['sample_id'].astype(str).values
).mean()

test_same_order = (
    test_df['sample_id'].astype(str).values
    == td_test['sample_id'].astype(str).values
).mean()

print('val same order:', val_same_order)
print('test same order:', test_same_order)

td val: (25858, 6)
td test: (25860, 6)
val same order: 1.0
test same order: 1.0


# MetaTox-inspired model

In [ ]:
# load zero-shot nli model
tokenizer = AutoTokenizer.from_pretrained(model_name)

model_dtype = torch.float16 if device == 'cuda' else torch.float32

mt_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    torch_dtype=model_dtype
)

mt_model = mt_model.to(device)
mt_model.eval()

print('loaded:', model_name)
print('id2label:', mt_model.config.id2label)
print('label2id:', mt_model.config.label2id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

loaded: MoritzLaurer/deberta-v3-base-zeroshot-v2.0
id2label: {0: 'entailment', 1: 'not_entailment'}
label2id: {'entailment': 0, 'not_entailment': 1}


In [ ]:
# find entailment label id
id2label = mt_model.config.id2label
label2id = mt_model.config.label2id

entail_id = None

# first try exact name
for name, idx in label2id.items():
    name_low = str(name).lower()

    if name_low == 'entailment':
        entail_id = int(idx)

# then try a safer backup search
if entail_id is None:
    for idx, name in id2label.items():
        name_low = str(name).lower()

        if 'entail' in name_low and 'not' not in name_low:
            entail_id = int(idx)

if entail_id is None:
    print('cannot find entailment id')
    print(id2label)
    print(label2id)
    raise ValueError('please check the model labels manually')

print('entailment id:', entail_id)
print('entailment label:', id2label[entail_id])

entailment id: 0
entailment label: entailment


In [ ]:
# hypotheses for semantic toxic judgement
toxic_hyp = 'This comment contains toxic, hateful, offensive, abusive, or harassing content.'
safe_hyp = 'This comment is normal, non-toxic, and not abusive.'

print('toxic hypothesis:')
print(toxic_hyp)

print('\nsafe hypothesis:')
print(safe_hyp)

toxic hypothesis:
This comment contains toxic, hateful, offensive, abusive, or harassing content.

safe hypothesis:
This comment is normal, non-toxic, and not abusive.


In [ ]:
# get metatox-inspired scores
# I compare toxic hypothesis with safe hypothesis

def get_mt_scores(text_list):
    score_list = []

    for start in tqdm(range(0, len(text_list), batch_size)):
        batch_texts = text_list[start:start + batch_size]

        premise_list = []
        hyp_list = []

        for text in batch_texts:
            premise_list.append(str(text))
            hyp_list.append(toxic_hyp)

            premise_list.append(str(text))
            hyp_list.append(safe_hyp)

        tokens = tokenizer(
            premise_list,
            hyp_list,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors='pt'
        )

        tokens = {
            k: v.to(device)
            for k, v in tokens.items()
        }

        with torch.no_grad():
            outputs = mt_model(**tokens)
            probs = F.softmax(outputs.logits.float(), dim=1)

        entail_probs = probs[:, entail_id].detach().cpu().numpy()
        entail_probs = entail_probs.reshape(-1, 2)

        toxic_score = entail_probs[:, 0]
        safe_score = entail_probs[:, 1]

        final_score = toxic_score / (toxic_score + safe_score + 1e-8)

        score_list.append(final_score)

    scores = np.concatenate(score_list, axis=0)

    return scores.astype('float32')

In [ ]:
# quick test before running the full data
small_texts = val_df['text'].head(5).tolist()

small_scores = get_mt_scores(small_texts)

print(small_scores)

  0%|          | 0/1 [00:00<?, ?it/s]

[3.0071610e-01 9.9996853e-01 8.9901549e-01 7.1582486e-05 9.9102966e-04]


# Validation

In [ ]:
# make or load validation scores
val_score_path = output_dir / 'mt_val_scores.npy'

if val_score_path.exists():
    print('load saved val scores')
    val_score = np.load(val_score_path)

else:
    print('make val scores')
    val_score = get_mt_scores(val_df['text'].tolist())
    np.save(val_score_path, val_score)

print('val scores:', val_score.shape)
print(pd.Series(val_score).describe())

make val scores


  0%|          | 0/1617 [00:00<?, ?it/s]

val scores: (25858,)
count    25858.000000
mean         0.199650
std          0.381064
min          0.000031
25%          0.000070
50%          0.000235
75%          0.047776
max          0.999974
dtype: float64


In [ ]:
# choose threshold on validation f1
y_val = val_df['label'].astype(int).values

best_t = 0.5
best_f1 = 0

for t in np.arange(0.1, 0.91, 0.05):
    val_pred = (val_score >= t).astype(int)
    now_f1 = f1_score(y_val, val_pred, zero_division=0)

    if now_f1 > best_f1:
        best_f1 = now_f1
        best_t = t

print('best threshold:', best_t)
print('best val f1:', best_f1)

best threshold: 0.9000000000000002
best val f1: 0.7586293146573286


In [ ]:
# make simple reasoning note
def make_reasoning(score):
    if score >= 0.7:
        return 'Zero-shot semantic judgement: the toxic hypothesis is strongly supported.'
    elif score <= 0.3:
        return 'Zero-shot semantic judgement: the non-toxic hypothesis is more supported.'
    else:
        return 'Zero-shot semantic judgement: the model is uncertain between toxic and non-toxic.'

In [ ]:
# save validation output
val_pred = (val_score >= best_t).astype(int)

mt_val_out = val_df.copy()
mt_val_out['mt_score'] = val_score
mt_val_out['mt_pred'] = val_pred
mt_val_out['reasoning'] = [make_reasoning(x) for x in val_score]

save_cols = [
    'sample_id',
    'text',
    'label',
    'source',
    'mt_score',
    'mt_pred',
    'reasoning'
]

mt_val_out = mt_val_out[save_cols].copy()

mt_val_out.to_csv(output_dir / 'metatox_inspired_val_outputs.csv', index=False)

print('saved validation output')
print(output_dir / 'metatox_inspired_val_outputs.csv')

mt_val_out.head()

saved validation output
/content/drive/MyDrive/Colab Notebooks/Project/metatox_inspired_zeroshot_outputs/metatox_inspired_val_outputs.csv


,sample_id,text,label,source,mt_score,mt_pred,reasoning
0,jg_36797,"""\n\nShall you immediately stop this sock-pupp...",0,jigsaw,0.301533,0,Zero-shot semantic judgement: the model is unc...
1,hx_validation_341,this is what <number> years of kike brainwashi...,1,hatexplain,0.999969,1,Zero-shot semantic judgement: the toxic hypoth...
2,hx_validation_1510,non asians btw u cant make jokes like “ why do...,0,hatexplain,0.899007,0,Zero-shot semantic judgement: the toxic hypoth...
3,jg_120449,I don't really think renaming should be a cent...,0,jigsaw,0.000071,0,Zero-shot semantic judgement: the non-toxic hy...
4,jg_112467,Apologies \n\nfor the stupid pointless bickeri...,0,jigsaw,0.000988,0,Zero-shot semantic judgement: the non-toxic hy...


# Test

In [ ]:
# make or load test scores
test_score_path = output_dir / 'mt_test_scores.npy'

if test_score_path.exists():
    print('load saved test scores')
    test_score = np.load(test_score_path)

else:
    print('make test scores')
    test_score = get_mt_scores(test_df['text'].tolist())
    np.save(test_score_path, test_score)

print('test scores:', test_score.shape)
print(pd.Series(test_score).describe())

make test scores


  0%|          | 0/1617 [00:00<?, ?it/s]

test scores: (25860,)
count    25860.000000
mean         0.196331
std          0.378715
min          0.000032
25%          0.000071
50%          0.000235
75%          0.043422
max          0.999974
dtype: float64


In [ ]:
# save test output
test_pred = (test_score >= best_t).astype(int)

mt_test_out = test_df.copy()
mt_test_out['mt_score'] = test_score
mt_test_out['mt_pred'] = test_pred
mt_test_out['reasoning'] = [make_reasoning(x) for x in test_score]

mt_test_out = mt_test_out[save_cols].copy()

mt_test_out.to_csv(output_dir / 'metatox_inspired_test_outputs.csv', index=False)

print('saved test output')
print(output_dir / 'metatox_inspired_test_outputs.csv')

mt_test_out.head()

saved test output
/content/drive/MyDrive/Colab Notebooks/Project/metatox_inspired_zeroshot_outputs/metatox_inspired_test_outputs.csv


,sample_id,text,label,source,mt_score,mt_pred,reasoning
0,jg_86226,"Yes, what is with these 'Impact on Fan Circles...",0,jigsaw,0.000524,0,Zero-shot semantic judgement: the non-toxic hy...
1,hx_test_341,<user> announced clerks <number> i am excited ...,0,hatexplain,0.000052,0,Zero-shot semantic judgement: the non-toxic hy...
2,hx_test_1510,goodmorning to toxic abusive women only,0,hatexplain,0.998002,1,Zero-shot semantic judgement: the toxic hypoth...
3,jg_132909,JACK OFF DUMMAY!!! \n\nstop lying to yourself ...,1,jigsaw,0.999951,1,Zero-shot semantic judgement: the toxic hypoth...
4,jg_61100,atrophysics and biological evolution,0,jigsaw,0.000106,0,Zero-shot semantic judgement: the non-toxic hy...


# Result

In [ ]:
# metric function
def get_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'false_positive_rate': fpr,
        'false_negative_rate': fnr,
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp)
    }

In [ ]:
# evaluate metatox-inspired branch
rows = []

val_metrics = get_metrics(
    mt_val_out['label'].astype(int).values,
    mt_val_out['mt_pred'].astype(int).values
)

val_row = {'method': 'metatox_inspired_zeroshot_val_mixed'}
val_row.update(val_metrics)
rows.append(val_row)


test_metrics = get_metrics(
    mt_test_out['label'].astype(int).values,
    mt_test_out['mt_pred'].astype(int).values
)

test_row = {'method': 'metatox_inspired_zeroshot_test_mixed'}
test_row.update(test_metrics)
rows.append(test_row)


for src in ['hatexplain', 'jigsaw']:
    part_df = mt_test_out[mt_test_out['source'] == src].copy()

    part_metrics = get_metrics(
        part_df['label'].astype(int).values,
        part_df['mt_pred'].astype(int).values
    )

    part_row = {'method': 'metatox_inspired_zeroshot_test_' + src}
    part_row.update(part_metrics)

    rows.append(part_row)


metric_df = pd.DataFrame(rows)

metric_df.to_csv(output_dir / 'metatox_inspired_metrics.csv', index=False)

metric_df

,method,accuracy,precision,recall,f1,false_positive_rate,false_negative_rate,tn,fp,fn,tp
0,metatox_inspired_zeroshot_val_mixed,0.925362,0.686044,0.848392,0.758629,0.062290,0.151608,20895,1388,542,3033
1,metatox_inspired_zeroshot_test_mixed,0.926566,0.691833,0.845638,0.761042,0.060447,0.154362,20937,1347,552,3024
2,metatox_inspired_zeroshot_test_hatexplain,0.710499,0.694611,0.914186,0.789414,0.586957,0.085814,323,459,98,1044
3,metatox_inspired_zeroshot_test_jigsaw,0.943934,0.690377,0.813476,0.746888,0.041298,0.186524,20614,888,454,1980


In [ ]:
# save setting
setting_df = pd.DataFrame({
    'item': [
        'model_name',
        'best_threshold',
        'toxic_hypothesis',
        'safe_hypothesis',
        'score_type',
        'note'
    ],
    'value': [
        model_name,
        str(best_t),
        toxic_hyp,
        safe_hyp,
        'toxic_entailment / (toxic_entailment + safe_entailment)',
        'MetaTox-inspired auxiliary detector, not full KG reproduction'
    ]
})

setting_df.to_csv(output_dir / 'metatox_inspired_setting.csv', index=False)

setting_df

,item,value
0,model_name,MoritzLaurer/deberta-v3-base-zeroshot-v2.0
1,best_threshold,0.9000000000000002
2,toxic_hypothesis,"This comment contains toxic, hateful, offensiv..."
3,safe_hypothesis,"This comment is normal, non-toxic, and not abu..."
4,score_type,toxic_entailment / (toxic_entailment + safe_en...
5,note,"MetaTox-inspired auxiliary detector, not full ..."


In [ ]:
# quick checks
print('val output:', mt_val_out.shape)
print('test output:', mt_test_out.shape)

print('\nscore summary')
print(mt_test_out['mt_score'].describe())

print('\nprediction count')
print(mt_test_out['mt_pred'].value_counts())

print('\nsource count')
print(mt_test_out['source'].value_counts())

val output: (25858, 7)
test output: (25860, 7)

score summary
count    25860.000000
mean         0.196331
std          0.378715
min          0.000032
25%          0.000071
50%          0.000235
75%          0.043422
max          0.999974
Name: mt_score, dtype: float64

prediction count
mt_pred
0    21489
1     4371
Name: count, dtype: int64

source count
source
jigsaw        23936
hatexplain     1924
Name: count, dtype: int64


In [ ]:
# check 02 saved files
for file in output_dir.iterdir():
    print(file.name)

mt_val_scores.npy
metatox_inspired_val_outputs.csv
mt_test_scores.npy
metatox_inspired_test_outputs.csv
metatox_inspired_metrics.csv
metatox_inspired_setting.csv
